In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation



datasets=[
    BNCI2014_008(),
    BNCI2014_009(),
    BNCI2015_003(),
    BI2012(),
    BI2013a(),
    BI2014a(),
    BI2014b(),
    BI2015a(),
    BI2015b(),
    Cattan2019_VR(),
    EPFLP300(),
    Huebner2017(),
    Huebner2018(),
    Lee2019_ERP()
]
paradigm = P300(
    resample=48,

)
cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

In [3]:
import copy
from sklearn.base import clone
import dask
import os
from sklearn.preprocessing import FunctionTransformer
import tensorly as tl

def eval_moabb_within_session(dataset, subject, pipe):
    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=False,
        random_state=42,
        n_jobs=5,
        suffix=f'bttda_dask_dataset-{dataset.code}_subject-{subject}_pipe-{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process({pipe:clone(pipelines[pipe])})





In [4]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd
from classification_erp import get_pipelines
from hpc import create_cluster, create_client, TIMEOUT

pipelines = get_pipelines()

with create_cluster(cluster='wice') as cluster, create_client(cluster) as client:
    results = []
    for dataset in datasets:
        print(f'Benchmarking on dataset {dataset.code}...')
        job_args = []
        for subject in dataset.subject_list:
            for pipe in pipelines.keys():
                job_args.append((dataset, subject,pipe))    
        with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
            results += Parallel(n_jobs=len(job_args), verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

Benchmarking on dataset BNCI2014-008...


[Parallel(n_jobs=24)]: Using backend DaskDistributedBackend with 459 concurrent workers.
[Parallel(n_jobs=24)]: Done   7 out of  24 | elapsed:   26.4s remaining:  1.1min
[Parallel(n_jobs=24)]: Done  24 out of  24 | elapsed:   27.1s finished
[Parallel(n_jobs=30)]: Using backend DaskDistributedBackend with 6552 concurrent workers.


Benchmarking on dataset BNCI2014-009...


[Parallel(n_jobs=30)]: Done   9 out of  30 | elapsed:    5.0s remaining:   11.7s
[Parallel(n_jobs=30)]: Done  30 out of  30 | elapsed:    5.1s finished
[Parallel(n_jobs=30)]: Using backend DaskDistributedBackend with 7776 concurrent workers.


Benchmarking on dataset BNCI2015-003...


...

[Parallel(n_jobs=39)]: Done  39 out of  39 | elapsed:    5.5s finished
[Parallel(n_jobs=36)]: Using backend DaskDistributedBackend with 9000 concurrent workers.


Benchmarking on dataset Huebner2018...


[Parallel(n_jobs=36)]: Done  19 out of  36 | elapsed:    4.5s remaining:    4.0s
[Parallel(n_jobs=36)]: Done  36 out of  36 | elapsed:    4.6s finished


Benchmarking on dataset Lee2019-ERP...


[Parallel(n_jobs=162)]: Using backend DaskDistributedBackend with 9000 concurrent workers.
[Parallel(n_jobs=162)]: Done  93 out of 162 | elapsed: 18.5min remaining: 13.7min
[Parallel(n_jobs=162)]: Done 162 out of 162 | elapsed: 35.2min finished
Exception in callback multi_future.<locals>.callback(<Task cancell...ession.py:77>>) at /data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.11/site-packages/tornado/gen.py:526
handle: <Handle multi_future.<locals>.callback(<Task cancell...ession.py:77>>) at /data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.11/site-packages/tornado/gen.py:526>
Traceback (most recent call last):
  File "/data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
  File "/data/leuven/352/vsc35289/miniconda3/envs/bttda/lib/python3.11/site-packages/tornado/gen.py", line 532, in callback
    result_list.append(f.result())
                       ^^^^^^^^^^
  Fil

In [5]:
results.to_csv('results/moabb_erp_new.csv')
results

,score,time,samples,subject,session,channels,n_sessions,dataset,pipeline
0,0.812390,10.808060,4200.0,1,0,8,1,BNCI2014-008,HODA
1,0.826712,19.842234,4200.0,1,0,8,1,BNCI2014-008,PARAFACDA
2,0.829833,21.622826,4200.0,1,0,8,1,BNCI2014-008,BTTDA
3,0.806786,12.670622,4200.0,2,0,8,1,BNCI2014-008,HODA
4,0.810290,16.947948,4200.0,2,0,8,1,BNCI2014-008,PARAFACDA
...,...,...,...,...,...,...,...,...,...
1963,0.951149,9.784805,4140.0,54,1,62,2,Lee2019-ERP,HODA
1964,0.977969,159.109116,4140.0,54,0,62,2,Lee2019-ERP,PARAFACDA
1965,0.971838,50.821396,4140.0,54,1,62,2,Lee2019-ERP,PARAFACDA
1966,0.978654,197.585617,4140.0,54,0,62,2,Lee2019-ERP,BTTDA


In [6]:
results = pd.read_csv('results/moabb_erp_new.csv')

In [7]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean')

dataset             pipeline 
BNCI2014-008        BTTDA        0.861996
                    HODA         0.852027
                    PARAFACDA    0.859946
BNCI2014-009        BTTDA        0.944046
                    HODA         0.932350
                    PARAFACDA    0.940348
BNCI2015-003        BTTDA        0.849860
                    HODA         0.827393
                    PARAFACDA    0.847095
BrainInvaders2012   BTTDA        0.912742
                    HODA         0.867444
                    PARAFACDA    0.907986
BrainInvaders2013a  BTTDA        0.920765
                    HODA         0.897757
                    PARAFACDA    0.917788
BrainInvaders2014a  BTTDA        0.877111
                    HODA         0.837611
                    PARAFACDA    0.872397
BrainInvaders2014b  BTTDA        0.904071
                    HODA         0.872314
                    PARAFACDA    0.899976
BrainInvaders2015a  BTTDA        0.934408
                    HODA         0.909565
    

In [8]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

pipeline
BTTDA        0.912475
HODA         0.888891
PARAFACDA    0.909393
Name: score, dtype: float64

In [9]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

pipeline,subject,session,channels,n_sessions,samples,dataset,BTTDA,HODA,PARAFACDA,score_diff
0,1,0,8,1,4200.0,BNCI2014-008,0.829833,0.812390,0.826712,0.017443
1,1,0,8,1,5400.0,BNCI2015-003,0.796895,0.758165,0.790870,0.038730
2,1,0,16,1,480.0,BrainInvaders2013a,0.981406,0.929688,0.983750,0.051719
3,1,0,16,1,768.0,BrainInvaders2012,0.947740,0.903440,0.949365,0.044301
4,1,0,16,1,1188.0,BrainInvaders2014a,0.960759,0.884267,0.950826,0.076492
...,...,...,...,...,...,...,...,...,...,...
651,60,0,16,1,792.0,BrainInvaders2014a,0.962693,0.914390,0.961925,0.048304
652,61,0,16,1,1368.0,BrainInvaders2014a,0.746962,0.707945,0.755164,0.039017
653,62,0,16,1,1367.0,BrainInvaders2014a,0.531261,0.551099,0.544938,-0.019839
654,63,0,16,1,420.0,BrainInvaders2014a,0.628163,0.636531,0.603878,-0.008367


In [10]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[.5, 1])
    fig.update_yaxes(range=[.5, 1])
    fig.add_shape(
        type="line",
        x0=0.5, y0=0.5, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig